# EDA: Train dataset (사기 공고 탐지)

목표:
- `training/data/processed/train.csv` 기준으로 데이터 분포/품질을 빠르게 파악
- (baseline 관점) **어떤 텍스트 n-gram이 사기(label=1) 쪽으로 작동하는지** 확인
- 숫자/메타성 피처(`text_len`, `has_salary` 등)가 라벨과 어떤 관계가 있는지 요약

주의:
- 이 노트북은 **EDA(설명/가설 탐색)** 용도입니다.
- 모델 성능 평가는 `training/evaluate_baseline.py`를 사용하세요(임계치 튜닝/지표 저장 포함).


In [7]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# 노트북은 실행 위치(cwd)가 사람/환경마다 달라서, 레포 루트를 자동으로 찾도록 합니다.
# 예: `training/notebooks`에서 실행하면 상대경로가 꼬여서 파일이 "없다"고 나올 수 있음

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())
DATA_PATH = REPO_ROOT / "training/data/processed/train.csv"
print("현재 작업 디렉토리:", Path.cwd())
print("REPO_ROOT:", REPO_ROOT)
print("DATA_PATH:", DATA_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} 파일을 찾을 수 없습니다. 먼저 레포 루트에서 `python3 training/preprocess.py`를 실행해 주세요."
    )

df = pd.read_csv(DATA_PATH)
print("✅ 로드 완료")
print("- shape:", df.shape)
print("- columns:", df.columns.tolist())


현재 작업 디렉토리: /Users/chanran/Github/samak-ai/training/notebooks
REPO_ROOT: /Users/chanran/Github/samak-ai
DATA_PATH: /Users/chanran/Github/samak-ai/training/data/processed/train.csv
✅ 로드 완료
- shape: (30994, 11)
- columns: ['text', 'fraudulent', 'salary_low', 'salary_high', 'salary_mid', 'has_salary', 'has_location', 'location_len', 'source', 'text_len', 'text_hash']


## 1) 라벨/소스 분포

- 현재 파이프라인 정책상 `test.csv`는 **RecruitmentScam만** 포함(라벨 섞임)
- `train.csv`는 RecruitmentScam train + (옵션) 증강 소스(LinkedIn_NegOnly / FakeJobPostings_PosOnly) 포함 가능


In [ ]:
assert "fraudulent" in df.columns, "fraudulent(label) 컬럼이 없습니다."
assert "source" in df.columns, "source 컬럼이 없습니다."

label_counts = df["fraudulent"].value_counts(dropna=False).sort_index()
print("라벨 분포(0=정상, 1=사기):")
display(label_counts)

by_source = (
    df.groupby(["source", "fraudulent"])["fraudulent"].count()
    .rename("n")
    .reset_index()
    .pivot(index="source", columns="fraudulent", values="n")
    .fillna(0)
    .astype(int)
)
by_source.columns = [f"label_{c}" for c in by_source.columns]
by_source["total"] = by_source.sum(axis=1)
by_source = by_source.sort_values("total", ascending=False)
print("\n소스별 라벨 분포:")
display(by_source)


SyntaxError: invalid syntax. Perhaps you forgot a comma? (2256585444.py, line 9)

## 2) 기본 품질 체크

- 텍스트 길이(`text_len`)가 너무 짧은 샘플이 많은지
- 결측/빈 텍스트가 있는지
- (옵션) 해시 중복 확인


In [ ]:
assert "text" in df.columns

empty_text = int((df["text"].fillna("").str.strip() == "").sum())
print("빈 텍스트 개수:", empty_text)

if "text_len" in df.columns:
    print("text_len 요약:")
    display(df["text_len"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

if "text_hash" in df.columns:
    dup = int(df.duplicated(subset=["text_hash"]).sum())
    print("text_hash 중복(동일 텍스트로 추정) 개수:", dup)


## 3) 숫자/메타성 피처 EDA

현재 `train.csv`에는 전처리에서 생성한 피처가 일부 포함돼 있어요:
- `has_salary`, `salary_low/high/mid`
- `has_location`, `location_len`
- `text_len`

MVP baseline은 텍스트 기반(TF-IDF) 위주지만, **라벨과 상관이 큰지** 정도는 빠르게 볼 수 있습니다.


In [ ]:
candidate_num_cols = [
    c
    for c in [
        "text_len",
        "has_salary",
        "salary_low",
        "salary_high",
        "salary_mid",
        "has_location",
        "location_len",
    ]
    if c in df.columns
]

print("사용 가능한 숫자 피처:", candidate_num_cols)

summary = df.groupby("fraudulent")[candidate_num_cols].agg(["mean", "median"]) if candidate_num_cols else None
if summary is not None:
    display(summary)


In [ ]:
if "text_len" in df.columns:
    plt.figure(figsize=(10, 4))
    sns.histplot(
        data=df,
        x="text_len",
        hue="fraudulent",
        bins=50,
        element="step",
        stat="density",
        common_norm=False,
    )
    plt.title("text_len 분포 (label별)")
    plt.xlim(0, np.nanpercentile(df["text_len"], 99))
    plt.show()

if "has_salary" in df.columns:
    plt.figure(figsize=(6, 4))
    sns.barplot(
        data=df,
        x="fraudulent",
        y="has_salary",
        estimator=np.mean,
        errorbar=None,
    )
    plt.title("급여 정보 포함 비율 (label별 평균)")
    plt.ylabel("P(has_salary=1)")
    plt.show()


## 4) 텍스트 피처(ngram)로 "사기"를 구분하는 단서 보기

TF-IDF + LogisticRegression을 **EDA 목적으로만** 빠르게 학습한 뒤,
계수(coef)를 통해 어떤 n-gram이 사기/정상에 기여하는지 확인합니다.

해석 팁:
- 계수가 클수록 `label=1(사기)`로 가는 방향
- 계수가 작을수록 `label=0(정상)`로 가는 방향
- 데이터셋 스타일/출처에 따른 편향도 있을 수 있으니, 소스 분포도 같이 보세요.


In [ ]:
text_col = "text"
y = df["fraudulent"].astype(int).to_numpy()
x_text = df[text_col].fillna("").astype(str).to_numpy()

# EDA 속도/메모리 보호: 너무 크면 샘플링
max_rows = 50000
if len(df) > max_rows:
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.choice(len(df), size=max_rows, replace=False)
    x_text = x_text[idx]
    y = y[idx]
    print(f"⚠️ 데이터가 커서 {max_rows}개로 샘플링했습니다.")

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
)
X = vectorizer.fit_transform(x_text)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
clf.fit(X, y)
print("✅ EDA용 TF-IDF + LR 학습 완료")


In [ ]:
feature_names = vectorizer.get_feature_names_out()
coef = clf.coef_[0]

def top_terms(n: int = 30):
    top_pos_idx = np.argsort(coef)[-n:][::-1]
    top_neg_idx = np.argsort(coef)[:n]

    pos = pd.DataFrame({"term": feature_names[top_pos_idx], "coef": coef[top_pos_idx]})
    neg = pd.DataFrame({"term": feature_names[top_neg_idx], "coef": coef[top_neg_idx]})
    return pos, neg

pos, neg = top_terms(40)
print("사기(label=1) 방향 상위 n-gram:")
display(pos)
print("\n정상(label=0) 방향 상위 n-gram:")
display(neg)


### (체크) 라벨 누수 키워드가 상위 피처로 튀어나오지 않는지

전처리에서 `mask_leakage=True`일 때 `fraudulent/scam/fake job` 같은 명백 누수 단어를 `<LEAK>`로 치환하도록 되어 있습니다.
여기서는 "상위 피처"에 그런 단어가 노출되는지 빠르게 확인합니다.


In [ ]:
leak_terms = {"fraudulent", "scam", "fake job", "not a scam", "<leak>"}

top_all = pd.concat([pos.head(80), neg.head(80)], ignore_index=True)
hits = []
for t in top_all["term"].astype(str):
    lt = t.lower()
    if any(k in lt for k in leak_terms):
        hits.append(t)

print("상위 피처 누수 키워드 히트:", hits)


## 5) 다음 액션 제안

- **Cross-dataset 테스트**(스타일 과적합 여부):
  - Train: FakeJobPostings / Test: RecruitmentScam
  - Train: RecruitmentScam / Test: FakeJobPostings
- LinkedIn/Pos-only 증강 비율(`linkedin_multiplier`, `fakepos_multiplier`)을 바꿔서 **top n-gram이 어떻게 바뀌는지** 비교
- `evaluate_baseline.py --tune`로 임계치(F1-max / precision 정책)를 저장하고, 서빙 응답(`modelPolicy`)에 그대로 노출되는지 확인
